In [4]:
from pyspark.sql.functions import when, col, udf
from pyspark.sql.types import StringType
# ensure the below library is installed on your fabric environment
import reverse_geocoder as rg

StatementMeta(, a8d94813-422e-4eff-8211-579f5f8f7e4d, 8, Finished, Available, Finished, False)

In [1]:
df = spark.read.table("earthquake_events_silver").filter(col('time') > start_date)

StatementMeta(, a8d94813-422e-4eff-8211-579f5f8f7e4d, 5, Finished, Available, Finished, False)

In [2]:
def get_country_code(lat, lon):
     coordinates = (float(lat), float(lon))
     return rg.search(coordinates)[0].get('cc')

StatementMeta(, a8d94813-422e-4eff-8211-579f5f8f7e4d, 6, Finished, Available, Finished, False)

In [5]:
# registering the udfs so they can be used on spark dataframes
get_country_code_udf = udf(get_country_code, StringType())
     


StatementMeta(, a8d94813-422e-4eff-8211-579f5f8f7e4d, 9, Finished, Available, Finished, False)

In [6]:
# adding country_code and city attributes
df_with_location = \
                df.\
                    withColumn("country_code", get_country_code_udf(col("latitude"), col("longitude")))
     

StatementMeta(, a8d94813-422e-4eff-8211-579f5f8f7e4d, 10, Finished, Available, Finished, False)

In [7]:
# adding significance classification
df_with_location_sig_class = \
                            df_with_location.\
                                withColumn('sig_class', 
                                            when(col("sig") < 100, "Low").\
                                            when((col("sig") >= 100) & (col("sig") < 500), "Moderate").\
                                            otherwise("High")
                                            )

StatementMeta(, a8d94813-422e-4eff-8211-579f5f8f7e4d, 11, Finished, Available, Finished, False)

In [8]:
# appending the data to the gold table
df_with_location_sig_class.write.mode('append').saveAsTable('earthquake_events_gold')
     

StatementMeta(, a8d94813-422e-4eff-8211-579f5f8f7e4d, 12, Finished, Available, Finished, False)